In [1]:

library(cubature)
library(MASS)
library(pracma)
library(mvtnorm)

library(LogConcDEAD)
library(logcondens) 
library(mclust)

Warning message:
“no DISPLAY variable so Tk is not available”
Package 'mclust' version 6.0.1
Type 'citation("mclust")' for citing this R package in publications.


Attaching package: ‘mclust’


The following object is masked from ‘package:mvtnorm’:

    dmvnorm




This notebook conducts a few tests, and checks results. The experiments are run from 'Experiments-241004-no-splitting.R

In [2]:
source('lcic.r')

# Check results

In [3]:
all_results <- list()
all_d <- c(5,10,15)
for (dind in 1:length(all_d)) {
    filename <- sprintf('Results/no_split_comparisons_over_n_d=%s.RData', all_d[dind])
    all_results[[dind]] <- readRDS(filename)
}

In [11]:
# as.data.frame(all_results[[1]])
# all_results[[1]]

# Convert to matrix

convert_to_matrix <- function(retrieved_comparison_data) {

    num_exp <- length(retrieved_comparison_data)
    results_mat <- matrix(nrow=num_exp ,ncol=7)
    colnames(results_mat) <- c("n", "d", "irep", "time_taken_logcondens", "time_taken_no_split", 
                              "hell_err_mean_logcondens", "hell_err_mean_no_split")
    
    for (eind in 1:num_exp){
        results_mat[eind, "n"] <- retrieved_comparison_data[[eind]]$n
        results_mat[eind, "d"] <- retrieved_comparison_data[[eind]]$d
        results_mat[eind, "irep"] <- retrieved_comparison_data[[eind]]$irep
        results_mat[eind, "time_taken_logcondens"] <- retrieved_comparison_data[[eind]]$time_taken_logcondens
        results_mat[eind, "time_taken_no_split"] <- retrieved_comparison_data[[eind]]$time_taken_no_split
        results_mat[eind, "hell_err_mean_logcondens"] <- retrieved_comparison_data[[eind]]$hell_err_mean_logcondens
        results_mat[eind, "hell_err_mean_no_split"] <- retrieved_comparison_data[[eind]]$hell_err_mean_no_split
    }
    return(results_mat)
    
}


In [24]:
get_ave_metric <- function(results_mat, all_n, metric_string){
    num_n_vals <- length(all_n)
    ave_metric_vec <- rep(0,num_n_vals)
    for (nvind in 1:num_n_vals){
        where_n <- (results_mat[,"n"]==all_n[nvind])
        ave_metric_vec[nvind] <- mean(results_mat[where_n, metric_string])
    }
    return(ave_metric_vec)
}

In [12]:
all_results_mats <- list()

for (dind in 1:3){
    all_results_mats[[dind]] <- convert_to_matrix(all_results[[dind]])
}

all_results_mats[[1]]

n,d,irep,time_taken_logcondens,time_taken_no_split,hell_err_mean_logcondens,hell_err_mean_no_split
100,5,1,0.05703282,0.09976006,0.101575924,0.111223862
100,5,2,0.03819418,0.05881500,0.095375612,0.070943948
100,5,3,0.02851319,0.04955578,0.122497673,0.090983612
100,5,4,0.04323387,0.05872798,0.098793210,0.083652976
100,5,5,0.04199553,0.04225636,0.112933099,0.094736040
500,5,1,0.11495757,0.17016315,0.027563206,0.020806935
500,5,2,0.09766936,0.15977645,0.024338836,0.020316461
500,5,3,0.11919379,0.18044662,0.027008243,0.018395854
500,5,4,0.12896991,0.15379286,0.032432028,0.024632088
500,5,5,0.12981844,0.16584945,0.030124905,0.020343320


In [32]:
pdf(file='Results/new-figures/compare-no-splitting.pdf', width=15, height=5)

options(repr.plot.width = 15, repr.plot.height = 5)
par(mfrow = c(1, 3), mar=c(6,6,4,2))

point_alpha <- 0.3
line_alpha <- 1
pch <- 19
lty <- 1

all_n <- c(100, 500, 1000, 2000, 3000)

for (dind in 1:3) {

     plot(x=all_results_mats[[dind]][,"n"], y=all_results_mats[[dind]][,"hell_err_mean_no_split"], pch=pch, col=rgb(1,0,0,point_alpha), main=bquote(d == .(all_d[dind])),
     cex.lab=2, cex.axis=1.5, cex.main=2.5, xlab=expression(n), ylab="Squared hellinger error", ylim=c(0,0.45))
     points(x=all_results_mats[[dind]][,"n"], y=all_results_mats[[dind]][,"hell_err_mean_logcondens"], pch=pch, col=rgb(0,0,1,point_alpha))
     lines(x=all_n, y=get_ave_metric(all_results_mats[[dind]], all_n, "hell_err_mean_no_split"), lty=lty, col=rgb(1,0,0,line_alpha))
     lines(x=all_n, y=get_ave_metric(all_results_mats[[dind]], all_n, "hell_err_mean_logcondens"), lty=lty, col=rgb(0,0,1,line_alpha))
     legend(x="topright", legend=c("With sample splitting", "Without sample splitting"), col=c(rgb(0,0,1,1), rgb(1,0,0,1)), lty=c(lty,lty), 
       cex=1.5, seg.len=3, y.intersp=1.5, bty="n")

}

dev.off()



png 
  2

In [33]:
# all_results_mats[[3]]
get_ave_metric(all_results_mats[[3]], all_n, "hell_err_mean_logcondens")

[1] 0.39965981 0.13496332 0.06890327 0.03793431 0.02906296

In [7]:
# Trial function. 

run_splitting_comparisons_over_n <- function(all_n, d, num_repeats_full, Sigma_max, eigensep, split_r, savefilename) {

    num_n <- length(all_n)
    num_exp <- num_n * num_repeats_full

    mu = rep(0,d)

    iexp <- 0
    all_results = list()
    
    # Print
    print("Num n values and experiment repeats:")
    print(num_n)
    print(num_repeats_full)
    flush.console()
    Sys.sleep(0.2)

    for (nind in 1:num_n) {
        for (irep in 1:num_repeats_full) {

            print('n index and repeat index:')
            print(nind)
            print(irep)
            flush.console()
            Sys.sleep(0.2)
            iexp <- iexp + 1
            
            n <- all_n[nind]

            # Simulate data
            SimData <- get_heteroskedastic_gaussian_data(d, n, true_mean_vec=mu, Sigma_max=Sigma_max, eigensep=eigensep)

            # Proposed estimator via logcondens: WITH sample splitting
            t_start <- Sys.time()
            my_estimator_logcondens <- generate_estimator_with_logcondens(SimData, r=split_r, plotting=FALSE)
            t_end <- Sys.time()
            time_taken_logcondens <- difftime(t_end, t_start, units="secs")

            # Proposed estimator via logcondens: WITHOUT sample splitting
            t_start <- Sys.time()
            my_estimator_no_split <- generate_estimator_with_logcondens(SimData, r=-1, plotting=FALSE)
            t_end <- Sys.time()
            time_taken_no_split <- difftime(t_end, t_start, units="secs")

            # Helper functions for computing sq hellinger errors
        
            hfun_logcondens <- function(X_samps){
                density_ratio <- evaluate_logcondens_estimator_vectorized(X_samps, my_estimator_logcondens)/heteroskedastic_gaussian_pdf_vectorized(X_samps, SimData)
                return(0.5*(sqrt(density_ratio)-1)^2)
            }
            
            
            hfun_no_split <- function(X_samps){
                density_ratio <- evaluate_logcondens_estimator_vectorized(X_samps, my_estimator_no_split)/heteroskedastic_gaussian_pdf_vectorized(X_samps, SimData)
                return(0.5*(sqrt(density_ratio)-1)^2)
            }
            
            generate_heteroskedastic_gaussian_samples_for_monte_carlo <- function(K_samps){
                return(mvrnorm(K_samps, mu=mu, Sigma=SimData$covariance_X))
            }

            # Error of proposed
            K_samps <- 10000
            # K_samps <- 1000
            num_repeats <- 50
            
            hellinger_error_estimate_statistics_logcondens <- naive_monte_carlo_integrate_repeated(hfun_logcondens, 
                                                                generate_heteroskedastic_gaussian_samples_for_monte_carlo, K_samps, num_repeats)
    
            
            hellinger_error_estimate_statistics_no_split <- naive_monte_carlo_integrate_repeated(hfun_no_split, 
                                                            generate_heteroskedastic_gaussian_samples_for_monte_carlo, K_samps, num_repeats)



            # Collect results
        
            exp_results_collect <- list("n"=n, "d"=d, "irep"=irep,
                                   "time_taken_logcondens"=time_taken_logcondens, "time_taken_no_split"=time_taken_no_split,
                                   "hell_err_mean_logcondens"=hellinger_error_estimate_statistics_logcondens$mean_val,
                                   "hell_err_mean_no_split"=hellinger_error_estimate_statistics_no_split$mean_val)

            all_results[[iexp]] <- exp_results_collect
        
            print("Done experiment num: ")
            print(iexp)
            print("################################################")
            flush.console()
            Sys.sleep(0.2)
        }
    }
    
    # Save
    # saveRDS(all_results, file=savefilename)

    return(all_results)
}


In [8]:
d <- 5
all_n <- c(1000)
Sigma_max <- 15
eigensep <- 1
num_repeats_full <- 1
split_r = 0.2

my_results_test <- run_splitting_comparisons_over_n(all_n, d, num_repeats_full, Sigma_max, eigensep, split_r, savefilename)


[1] "Num n values and experiment repeats:"
[1] 1
[1] 1
[1] "n index and repeat index:"
[1] 1
[1] 1
[1] "Diagonal covariance: "
     [,1] [,2] [,3] [,4] [,5]
[1,]   15    0    0    0    0
[2,]    0   14    0    0    0
[3,]    0    0   13    0    0
[4,]    0    0    0   12    0
[5,]    0    0    0    0   11
[1] "PCA done!"
[1] "Marginal: "
[1] 1
[1] "Marginal: "
[1] 2
[1] "Marginal: "
[1] 3
[1] "Marginal: "
[1] 4
[1] "Marginal: "
[1] 5
[1] "NOTE: r is -1. Will not split samples!"
[1] "PCA done!"
[1] "Marginal: "
[1] 1
[1] "Marginal: "
[1] 2
[1] "Marginal: "
[1] 3
[1] "Marginal: "
[1] 4
[1] "Marginal: "
[1] 5
[1] "Done experiment num: "
[1] 1
[1] "################################################"


In [12]:
sprintf("Hi%s.txt", 10)

[1] "Hi10.txt"